In [ ]:
%cd ..

In [ ]:
from collections import defaultdict
from tqdm import tqdm

import numpy as np
import jax
import jax.numpy as jnp
from jax.sharding import SingleDeviceSharding

import optax
import orbax
from orbax import checkpoint

import flax.nnx as nnx
from flax.training.early_stopping import EarlyStopping

from aptax.dataset import TablesDataset, create_dataloader
from aptax.llm import MiniTabPFN

In [ ]:
num_epochs = 100
batch_size = 8
dataset = TablesDataset(
    n_tables=100,
    n_samples=100,
    n_features=5,
    n_classes=3,
)
dataloader, batches_per_epoch = create_dataloader(dataset, batch_size=batch_size, num_epochs=1)

total_steps = num_epochs * batches_per_epoch
warmup_steps = max(1, total_steps // 10)
print(f"Dataset size: {len(dataset)}")
print(f"Batches per epoch: {batches_per_epoch}")
print(f"Total training steps: {total_steps:,}")
print(f"Warmup steps: {warmup_steps:,}")

In [ ]:
embed_dim = 192
num_heads = 6
output_dim = 3
num_transformer_blocks = 3

model = MiniTabPFN(
    embed_dim=embed_dim,
    output_dim=output_dim,
    num_heads=num_heads,
    num_transformer_blocks=num_transformer_blocks,
    rngs=nnx.Rngs(0),
)
print(model)

In [ ]:
# @nnx.jit
def train_step(model, optimizer, batch):
    inputs, targets, train_test_split_index = batch

    def loss_fn(model):
        logits = model(
            x=inputs, y=targets, train_test_split_index=train_test_split_index
        )
        loss = optax.softmax_cross_entropy(
            logits=logits, labels=targets
        ).mean()
        return loss, logits

    (loss, logits), grads = nnx.value_and_grad(loss_fn, has_aux=True)(model)
    optimizer.update(grads)
    return loss, logits


lr_schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=5e-4,
    warmup_steps=10,
    decay_steps=total_steps,
    end_value=5e-5,
)
optimizer = nnx.ModelAndOptimizer(
    model,
    optax.adamw(learning_rate=lr_schedule, weight_decay=0.01),
)
metrics = nnx.MultiMetric(
    loss=nnx.metrics.Average("loss"),
)

In [ ]:
step = 0
score_history = defaultdict(list)
early_stop = EarlyStopping(min_delta=1e-2, patience=5)
for epoch in range(num_epochs):
    epoch_history = defaultdict(list)
    batch_idx = 0
    for batch in (pbar := tqdm(dataloader, total=batches_per_epoch)):
        inputs = jnp.array(batch["inputs"])
        labels = jnp.array(batch["labels"])
        labels = jnp.array(nnx.one_hot(labels, num_classes=jnp.max(jnp.unique(labels))+1), dtype=jnp.float32)

        train_test_split_index = jnp.array(len(dataset))
        loss, logits = train_step(
            model, optimizer, (inputs, jnp.atleast_3d(labels), train_test_split_index)
        )
        metrics.update(loss=loss, logits=logits, labels=labels)

        for metric, value in metrics.compute().items():
            epoch_history[f"train_{metric}"].append(value)
        metrics.reset()

        current_lr = lr_schedule(step)
        pbar.set_description(
            f"epoch: {epoch + 1}/{num_epochs}, lr: {current_lr:.2e}, "
            f"loss: {np.mean(epoch_history['train_loss']):.3f}"
        )
        step += 1
        batch_idx += 1

    for metric, values in epoch_history.items():
        score_history[metric].append(values)
    early_stop = early_stop.update(np.mean(score_history["train_loss"]))
    if early_stop.should_stop:
        print(f"\nMet early stopping criteria! Breaking at epoch {epoch}.")
        break